# Notebook for Generating Captions

This notebook is used to generate artificial captions and create a synthetic dataset which paraphrases upon the captions in the pixel_prose dataset.

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from json import JSONDecodeError
import csv
import warnings


from dotenv import load_dotenv
from openai import OpenAI, RateLimitError

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [3]:
df = pd.read_csv("pixel_prose.csv")

In [4]:
df["original_caption"]

0      Car riding on wet street. Car driving on wet c...
1                             Water Coming Out Of A Pipe
2      MARCH 2009 ''It's sad that a boy and girl can ...
3             Humpback Whale tail in the Bay of Banderas
4      The gaming Team logo of team DUO Team Logo, My...
                             ...                        
494    The extended body on the this fly creates a mo...
495    Vector illustration of flowers in the shape of...
496    Winter snow storm in Boston's Public Garden .....
497    Close-up of business plan with people working ...
498    All smiles, <PERSON> poses for the lens of pho...
Name: original_caption, Length: 499, dtype: object

In [5]:
df.index[0]

0

In [6]:
df["vlm_caption"][1]

' This image displays: A large rusty pipe with a white ring near the bottom. Water is gushing out of the pipe into a murky body of water below. The pipe is attached to a large hopper or funnel that is painted white and has some rust. The hopper is mostly obscured by a metal fence with square holes in it. There is a single plant growing next to the fence, with bright green leaves. The background is a blur of more metal fencing and concrete. The image is a photograph that has been taken from a low angle so that the water seems to be gushing towards the viewer.'

In [7]:
all_short_captions = []
for idx in df.index:
    all_short_captions.append({idx: df["original_caption"][idx]})

In [11]:
df

,Unnamed: 0,uid,url,key,status,original_caption,vlm_model,vlm_caption,toxicity,severe_toxicity,...,width,height,original_width,original_height,exif,sha256,image_id,author,subreddit,score
0,1,c9d31cce99482e8da13f6a868e2472c3bc3ea05a874e5b...,https://thumbs.dreamstime.com/b/car-riding-wet...,7740001,success,Car riding on wet street. Car driving on wet c...,gemini-pro-vision,This image displays:\n\nA black car driving t...,0.005905,0.000017,...,800,534,800,534,{},c9d31cce99482e8da13f6a868e2472c3bc3ea05a874e5b...,NaN,NaN,-1,-1
1,2,f3a6ad643814199a98f872229758b8e735773c1803e2f1...,https://images.pexels.com/photos/770224/pexels...,7740009,success,Water Coming Out Of A Pipe,gemini-pro-vision,This image displays: A large rusty pipe with ...,0.002487,0.000007,...,500,750,500,750,{},f3a6ad643814199a98f872229758b8e735773c1803e2f1...,NaN,NaN,-1,-1
2,3,3b03f5e3a32499281dbeebb9a0e1d3dfe883bc7b72e2eb...,https://filmfare.wwmindia.com/content/2017/jul...,7740010,success,MARCH 2009 ''It's sad that a boy and girl can ...,gemini-pro-vision,This image displays: three pictures of Katrin...,0.001743,0.000003,...,600,450,600,450,{},3b03f5e3a32499281dbeebb9a0e1d3dfe883bc7b72e2eb...,NaN,NaN,-1,-1
3,4,9f3b5395744728e68b85083c0411c3b4139214f1258bf7...,https://southernersays.com/wp-content/uploads/...,7740000,success,Humpback Whale tail in the Bay of Banderas,gemini-pro-vision,This image displays:\nA photograph of a whale...,0.006152,0.000024,...,1024,662,1024,662,{},9f3b5395744728e68b85083c0411c3b4139214f1258bf7...,NaN,NaN,-1,-1
4,5,e3c7435ce411fea5656c792355a3811163da286673f8e5...,https://i.pinimg.com/originals/9a/f5/a6/9af5a6...,7740035,success,"The gaming Team logo of team DUO Team Logo, My...",gemini-pro-vision,"This image displays:\n\nText that reads ""20% ...",0.007788,0.000023,...,1820,1024,1920,1080,{},e3c7435ce411fea5656c792355a3811163da286673f8e5...,NaN,NaN,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494,594,34aa54a20f7a91d93af20ac4183e1fe8ef19d57284c353...,https://images.squarespace-cdn.com/content/v1/...,7740813,success,The extended body on the this fly creates a mo...,gemini-pro-vision,This image displays: A close-up photograph of...,0.000551,0.000001,...,1000,750,1000,750,{},34aa54a20f7a91d93af20ac4183e1fe8ef19d57284c353...,NaN,NaN,-1,-1
495,595,4d315306896f25b3fc3b1721a9e8a265b70818b5d22695...,https://image.shutterstock.com/z/stock-vector-...,7740791,success,Vector illustration of flowers in the shape of...,gemini-pro-vision,This image displays:\n\nA black crescent moon...,0.007320,0.000019,...,1024,1092,1500,1600,"{""Image Orientation"": ""Horizontal (normal)"", ""...",4d315306896f25b3fc3b1721a9e8a265b70818b5d22695...,NaN,NaN,-1,-1
496,596,7bdc3ead9988c533e52650eb9f374860b57b452b208e74...,https://www.boston-discovery-guide.com/image-f...,7740821,success,Winter snow storm in Boston's Public Garden .....,gemini-pro-vision,This image displays: Two women walking away f...,0.002525,0.000017,...,800,440,800,440,"{""Image Make"": ""Panasonic"", ""Image Model"": ""DM...",7bdc3ead9988c533e52650eb9f374860b57b452b208e74...,NaN,NaN,-1,-1
497,598,77fe9e8070fe07894f8deb74cc64cfb3afa72829b2f5c9...,https://image.freepik.com/free-photo/close-up-...,7740793,success,Close-up of business plan with people working ...,gemini-pro-vision,This image displays:\n\nIn the foreground of ...,0.000409,0.000001,...,626,433,626,433,{},77fe9e8070fe07894f8deb74cc64cfb3afa72829b2f5c9...,NaN,NaN,-1,-1


In [ ]:
df.loc[[0,1,2,3,4]]["url"]

0    https://thumbs.dreamstime.com/b/car-riding-wet...
1    https://images.pexels.com/photos/770224/pexels...
2    https://filmfare.wwmindia.com/content/2017/jul...
4    https://i.pinimg.com/originals/9a/f5/a6/9af5a6...
3    https://southernersays.com/wp-content/uploads/...
Name: url, dtype: object

In [ ]:
def initialize_csv(file_path):
    """
    initialises a CSV 

    Args:
        file_path: a string which will be the location of the CSV
    Return:
        None
    """
    with open(file_path, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["id", "img_url", "original_caption", "paraphrased_caption"])


def append_to_csv(file_path, rows):
    """
    Adds multiple rows to the CSV
    Args:
        file_path: a string which will be the location of the CSV
        rows: a list containing tuples where each tuple will be a row
    """
    with open(file_path, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerows(rows)



def create_artificial_captions(captions):
    """
    Generates artificial captions using OpenAIs GPT

    Args:
        captions: a list of objects where each object has a key (id) and value (caption)
    Return:
        output_dict: A dictionary where key is the id and value is the paraphrased caption
    """
    print(captions)
    # define model and temperature. A higher temperature is chosen for variability in the captions so that it is not the same caption just reworded
    MODEL = "gpt-4o-mini"
    TEMPERATURE = 1.0

    prompt = f"""
        You are a helpful assistant that rewrites short image captions. You are generating captions to assess how well image retrieval can go with varying captions. Paraphrase the following captions. You may deviate slightly from the text to make the output caption slightly more different. You may omit less relevant details such as names of places or people in some captions. DO NOT ADD ANYTHING MORE THAN A REPHRASED CAPTION. Do NOT reorder the generated captions. Return each generated caption in the same order that they were passed in. Respond only with a JSON object. Use double quotes for all keys and values where the key is the id and the value is the rephrased caption. The id is given as part of each caption. You are to make this same id the key in your output. It is imperitive that you do not reorder, or skip over the captions. \n\n\n

        {captions}
        """
    
    try:

        # send a request
        response = client.chat.completions.create(
            model=MODEL,
            temperature=TEMPERATURE,
            messages=[
                {"role": "system", "content": "You rephrase lists of short captions without changing meaning."},
                {"role": "user", "content": prompt}
            ]
        )

        # extract response and to generated_original_captions list
        raw_output = response.choices[0].message.content.strip()
        print(f"Raw output is {raw_output}")
        if raw_output.startswith("```"):
            lines = raw_output.splitlines()
            # remove the first and last lines (```json and ```) as it causes errors otherwise
            raw_output = "\n".join(lines[1:-1])
        output_dict = json.loads(raw_output)

        return output_dict

    
    except RateLimitError as e:
        warnings.warn(e)
    except JSONDecodeError as e:
        warnings.warn(e)

def generate_captions(df, col_name, path_to_save, num_partitions):
    """
    Generates artificial captions by preprocessing dataframe, extracting column and calling create_artificial_captions

    Args:
        df: A dataframe containing the captions to paraphrase
        col_name: A string referencing the column name of the captions (should be original_caption or vlm_model)
        path_to_save: a string which defines where to save the rows containing the paraphrased captions to
        num_partitions: An integer used to control number of partitions to split dataframe into. This is the number of requests made to the API and is needed so that the API will not make too many mistakes after taking the whole dataframe in.

    Return:
        None

    Note: We have found that increasing num_partitions reduces risk of error, likely because there is less to process per call
    """
    
    # create a list of dictionaries where key is id and value is the caption to convert
    all_short_captions = []
    for idx in df.index:
        all_short_captions.append({idx: df[col_name][idx]})

    # process x amount of captions first, save, then add x more and so on
    num_captions_per_batch = (len(all_short_captions) // num_partitions) + 1
    print(num_captions_per_batch)

    # we want to process the first x captions, and save it to a csv
    for partition in range(num_partitions):
        output_dict = {}
        if partition == num_partitions - 1:
            # take all remaining
            output_dict = create_artificial_captions(all_short_captions[partition * num_captions_per_batch: ])
        else: 
            # take exactly a batch worth of captions
            output_dict = create_artificial_captions(all_short_captions[partition * num_captions_per_batch: (partition+1)*num_captions_per_batch])
        print("Output dict is ", output_dict)

        # now save it to the csv by making a valid row
        # convert the string ids into int
        ids = list(map(int, output_dict.keys()))
        # make sure that the captions are in the correct order
        captions = [output_dict[str(i)] for i in ids]
        img_urls = df.loc[ids]["url"].tolist()
        og_captions = df.loc[ids][col_name].tolist()
        rows = list(zip(ids, img_urls, og_captions, captions))

        # add to csv
        append_to_csv(path_to_save, rows)
        
        

## Generating Short Captions

In [ ]:
FILE_PATH = "./short_generated_captions.csv"

initialize_csv(FILE_PATH)

In [ ]:
generate_captions(df, "original_caption", FILE_PATH, 50)

10
[{0: 'Car riding on wet street. Car driving on wet city street during a downpour stock photography'}, {1: 'Water Coming Out Of A Pipe'}, {2: "MARCH 2009 ''It's sad that a boy and girl can never be friends out here. First they linked me with <PERSON>, it's <PERSON>'s turn now.'' Katrina: It's true. They're still doing it. AUGUST 2009 '' <PERSON>, <PERSON>"}, {3: 'Humpback Whale tail in the Bay of Banderas'}, {4: 'The gaming Team logo of team DUO Team Logo, My Design, Gaming, Logos, Game, Videogames, Logo, Games'}, {5: 'Tea, chilies, and takeaway: what food choices reveal about British Muslim identity'}, {6: 'The Bath Co. Camberley satin grey vanity unit with basin White Vanity Unit, Basin Vanity Unit, Gray Vanity, Bathroom Vanity Units, Vanity Sink, Bathroom Basin, Family Bathroom, Downstairs Bathroom, Small Bathroom'}, {7: 'Celebs attend the launch of a style store'}, {8: 'Europe always amazes me on how well their architecture withstands the test of time. Croatia was no exception.'}

## Generate Long Captions

In [ ]:
LONG_CAPTIONS_FILE_PATH = "./long_generated_captions.csv"
initialize_csv(LONG_CAPTIONS_FILE_PATH)

generate_captions(df, "vlm_model", LONG_CAPTIONS_FILE_PATH, 50)